> **Offline edition.** This is BlueQubit's *Out-of-time-order correlators (OTOCs): a hands-on tutorial*,
> unchanged except for: (1) Part 5 runs on `LocalBQ` (offline exact / MPS / Pauli-path) instead of the
> cloud, and (2) ✏️ exercises with self-checks after each part. The original is in
> `../originals/otoc-intro.ipynb`. **This is the most important notebook for the hackathon.**

# Out-of-time-order correlators (OTOCs): a hands-on tutorial

This notebook is a self-contained introduction to out-of-time-order correlators: what they measure, how to compute them, and why they're hard to simulate. It mixes short explanations with code you can run and change.

By the end you'll be able to:

1. explain what OTOC(1) and OTOC(2) measure,
2. build the echo circuits that measure them,
3. check an estimate against an exact answer, and measure how well a set of estimates tracks the truth,
4. run exact simulations, and approximate ones (matrix product states (MPS) or Pauli paths), on BlueQubit.

You need some linear algebra (vectors, matrices, the conjugate transpose) and a little familiarity with qubits and quantum gates. Everything else is explained as it comes up.

## Useful reading

* B. Swingle, [Unscrambling the physics of out-of-time-order correlators](https://doi.org/10.1038/s41567-018-0295-5), Nature Physics 14, 988 (2018). A short, readable overview.
* Google Quantum AI and collaborators, [Observation of constructive interference at the edge of quantum ergodicity](https://doi.org/10.1038/s41586-025-09526-6), Nature (2025). The "Quantum Echoes" experiment, which measured OTOC(2) on Google's Willow chip.

---
# Part 0: Setup (offline)

Everything you need is already in this folder: `otoc_tools.py` (BlueQubit's helpers),
`offline_sim.py` (offline simulators) and `checks.py`. Pick the **Python (BluQubit)** kernel and run the cell below.

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Check that the packages from Part 0 are installed.
try:
    import numpy as np
    import matplotlib.pyplot as plt
    import qiskit
except ModuleNotFoundError as err:
    raise SystemExit(
        f"Missing package: {err.name}. This notebook is running on the Python at\n    {sys.executable}\n"
        "which doesn't have the packages this notebook needs. Fix: install them (see Part 0), "
        "then pick that environment's kernel (VS Code: Select Kernel, top right) and restart the kernel."
    ) from None
try:
    import importlib, otoc_tools
    importlib.reload(otoc_tools)   # pick up any edits to otoc_tools.py without restarting the kernel
    from otoc_tools import (brickwork, to_qiskit, echo_circuit, exact_otoc1, exact_otoc2,
                            simulate_expz, z_label, load_qasm, score, write_csv, read_csv,
                            zero, run_U, run_Udag, apply1, expZ, X, Z, draw, echo_schematic,
                            near_clifford_circuit)
except ModuleNotFoundError:
    raise SystemExit("Can't find otoc_tools.py. Open this notebook from the folder that contains "
                     "otoc_tools.py, so they sit next to each other.") from None

rng = np.random.default_rng(2026)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print(f"Ready: Python {sys.version.split()[0]}, numpy {np.__version__}, qiskit {qiskit.__version__}")

from offline_sim import LocalBQ, z_sum, pauli_path_expectation
import checks
bq = LocalBQ()

---
# Part 1: What is an OTOC?

## The quantum butterfly effect

You've probably heard of the butterfly effect: a butterfly flaps its wings and, weeks later, the weather somewhere else is different. Quantum systems have a version of this, and OTOCs measure it.

Picture a row of qubits. We nudge qubit $b$ by applying a Pauli operator $B$, for example $X_b$. Physicists call this nudge the **butterfly**. Then the qubits evolve under a circuit $U$.

The easiest way to follow the nudge is to let the operator evolve instead of the state (the **Heisenberg picture**). After the circuit, the nudge has become

$$B(t) = U^\dagger B\, U .$$

At the start, $B$ only touches qubit $b$. As gates act, $B(t)$ grows into a sum of Pauli strings that cover more and more qubits.

> **Notation used from here on.**
>
> * **Commutator.** For two operators $V$ and $W$, the **commutator** is $[V, W] = VW - WV$. It measures whether the order you apply them in matters. If $[V, W] = 0$ we say $V$ and $W$ **commute**. Operators acting on different qubits always commute: for example $[X_0, Z_1] = X_0 Z_1 - Z_1 X_0 = 0$. On the same qubit they usually don't: $XZ = -iY$ and $ZX = iY$, so $[X, Z] = -2iY \neq 0$.
> * **Dagger.** $V^\dagger$ is the conjugate transpose of $V$. A unitary satisfies $U^\dagger U = I$, so $U^\dagger$ undoes $U$ (runs the circuit backward). An operator with $V^\dagger = V$ is **Hermitian**. All Pauli operators are both Hermitian and unitary.
> * **Angle brackets.** $\langle V\rangle$ means the expectation value of $V$ in the starting state, $\langle V\rangle = \langle 0\cdots0|\,V\,|0\cdots0\rangle$.
> * **Pauli string.** A tensor product of single-qubit Paulis, one per qubit, such as $X_0 Z_3$ (meaning $X$ on qubit 0, $Z$ on qubit 3, identity elsewhere).

To check whether the nudge has reached some far-away qubit $q$, we use a **probe** $M = Z_q$ and ask whether $B(t)$ and $M$ still commute. Before the nudge arrives they act on different qubits, so $[B(t), M] = 0$. Once it arrives they stop commuting. To measure how much, we take the "size" of the commutator, the expectation value of $[B(t),M]^\dagger[B(t),M]$ (which is never negative):

$$\mathcal C = \tfrac12\big\langle\, [B(t),M]^\dagger[B(t),M]\,\big\rangle \;=\; 1 - \mathrm{Re}\,\langle B(t)\,M\,B(t)\,M\rangle .$$

Here's why the two sides are equal. Write $A = B(t)$ for short. Both $A$ and $M$ are Hermitian, and both square to the identity ($A^2 = U^\dagger B^2 U = I$ and $M^2 = I$, because Paulis square to $I$). Then $[A,M]^\dagger = MA - AM$, and multiplying out,

$$[A,M]^\dagger[A,M] = (MA - AM)(AM - MA) = M A^2 M - MAMA - AMAM + A M^2 A = 2 - MAMA - AMAM .$$

Finally, $MAMA = (AMAM)^\dagger$, so the expectation values of those two terms are complex conjugates and add up to $2\,\mathrm{Re}\,\langle AMAM\rangle$. Dividing by 2 gives the formula. The quantity on the right,

$$F = \langle B(t)\,M\,B(t)\,M\rangle ,$$

is the **OTOC**.

> **Key idea.** If $F \approx 1$, the butterfly hasn't reached qubit $q$ yet. If $F \approx 0$, its effect has spread over the whole system, and we say the information has been **scrambled**.

**Where does the name come from?** Operators act on a state from the right, so a product like $\langle V\,W\rangle$ is read from right to left: first $W$ acts, then $V$. In an ordinary experiment you nudge the system at time 0 and look at it at a later time $t$. That gives a correlator like $\langle B(t)\,M\rangle$, whose times, read right to left, go $0$ then $t$. They follow the clock, so it's called **time-ordered**, and one forward run of the experiment measures it.

Why is $B(t)$ at time $t$ but $M$ at time 0? In the Heisenberg picture, an operator "at time $t$" is one that has been sandwiched by the circuit: $V(t) = U^\dagger V\,U$. $B(t)$ has this sandwich around it. $M = Z_q$ appears on its own, with no $U$ around it, so it is the probe as it was at the start, before any gates ran. That's time 0.

Now read $F = \langle B(t)\,M\,B(t)\,M\rangle$ the same way. The times go $0$, then $t$, then back to $0$, then $t$ again. The clock has to jump backward in the middle, so the operators are **out of time order**, and that's where the name comes from.

You can see the jumps by writing out $B(t) = U^\dagger B\,U$:

$$F = \langle 0|\;U^\dagger B\,U\;M\;U^\dagger B\,U\;M\;|0\rangle .$$

Read right to left, $U$ takes the system forward from time 0 to time $t$ and $U^\dagger$ takes it back from $t$ to 0. So $F$ means: forward, backward, forward, backward. An experiment that can only let time run forward can't do this. A quantum computer can, because running backward just means applying the gates of $U^\dagger$.

## From formula to circuit: the echo

A real quantum computer can only start in $|0\cdots0\rangle$, apply gates and measure. Two small observations turn $F$ into exactly that.

First, $Z_q$ does nothing to $|0\cdots0\rangle$, so the right-most $M$ in $F$ can be dropped. Second, $B(t)$ is Hermitian, so

$$\langle 0|\,B(t)\,M\,B(t)\,|0\rangle = \langle\psi|M|\psi\rangle \qquad\text{with}\qquad |\psi\rangle = U^\dagger B\, U|0\rangle .$$

So the OTOC is just **the value of $\langle Z_q\rangle$ at the end of this circuit**:

### ✏️ Exercise: check the OTOC identity numerically

Part 1 claims $\mathcal C = \tfrac12\langle [A,M]^\dagger[A,M]\rangle = 1 - \mathrm{Re}\,F$ with
$A = B(t) = U^\dagger B U$ and $F = \langle A M A M\rangle$. Verify it on 3 qubits with a random $U$.

Use full matrices: `U` (8×8), `B` = $X$ on qubit 0, `M` = $Z$ on qubit 2 (Qiskit order:
`np.kron(Z, np.kron(I, I))` is Z on qubit 2). Set `C_lhs` and `C_rhs` (both real numbers) and
`F` (complex). Expectation values are taken in $|000\rangle$ (the vector `e0` below).

In [ ]:
from scipy.stats import unitary_group
U = unitary_group.rvs(8, random_state=1)
I2 = np.eye(2)
e0 = np.zeros(8); e0[0] = 1
B = np.kron(I2, np.kron(I2, X))      # X on qubit 0
M = np.kron(Z, np.kron(I2, I2))      # Z on qubit 2

F = None       # YOUR CODE
C_lhs = None
C_rhs = None

# --- check (don't edit) ---
checks.truthy("C_lhs == C_rhs", C_lhs is not None and abs(C_lhs - C_rhs) < 1e-10,
              hint="A = U^dag B U; C_lhs = <comm^dag comm>/2; C_rhs = 1 - Re F", todo=C_lhs is None)
checks.truthy("0 <= C <= 2", C_lhs is not None and -1e-12 <= C_lhs <= 2 + 1e-12, todo=C_lhs is None)

In [ ]:
# The OTOC(1) echo, drawn with U and U† as single boxes:
# X on qubit b = 0 is the butterfly, and the meter on qubit q = 5 reads <Z_q>.
draw(echo_schematic(n=6, b=0, q=5, order=1))

This is called an **echo**: run the circuit forward, nudge one qubit, then run the circuit backward. If you leave out the nudge, $U^\dagger U = I$ and every qubit comes back to $|0\rangle$, so every $\langle Z_q\rangle = 1$. With the nudge, the qubits that the butterfly has reached don't come back cleanly. We'll call this quantity **OTOC(1)**, because Part 3 introduces a second-order version.

For $U$ we use a **brickwork** circuit: layers of random two-qubit gates on neighbouring pairs, offset by one qubit from layer to layer like bricks in a wall. Here's an echo on 6 qubits with 4 layers.

In [ ]:
n, depth, b = 6, 4, 0
gates = brickwork(n, depth, rng)          # list of (layer, i, 4x4 random unitary on qubits i, i+1)

# A picture of the echo. Each box is one random two-qubit gate, labelled by its layer t.
from qiskit import QuantumCircuit
from qiskit.circuit import Gate
echo_pic = QuantumCircuit(n)
for t, i, G in gates:                                   # U: layer by layer
    echo_pic.append(Gate("G", 2, [], label=f"$G_{t}$"), [i, i + 1])
echo_pic.barrier(); echo_pic.x(b); echo_pic.barrier()  # the butterfly
for t, i, G in reversed(gates):                         # U†: the same gates inverted, in reverse order
    echo_pic.append(Gate("Gdg", 2, [], label=f"$G_{t}^\\dagger$"), [i, i + 1])
draw(echo_pic)

## What $\langle Z_q\rangle$ means

Running a circuit produces a statevector $|\psi\rangle$: one complex **amplitude** $\psi_x$ for every bitstring $x$, so $2^n$ numbers in total. If you measured all the qubits, you would see bitstring $x$ with probability $|\psi_x|^2$.

$\langle Z_q\rangle$ squeezes all of that down to one number about qubit $q$. There are two ways to read it, and they give the same answer.

**As an average over measurements.** Measure the qubits. Record $+1$ if qubit $q$ reads 0 and $-1$ if it reads 1. Repeat many times and average the records. The **expectation value** $\langle Z_q\rangle$ is what that average tends to after very many repeats:

$$\langle Z_q\rangle = \sum_x |\psi_x|^2\,(-1)^{x_q} = P(\text{qubit } q \text{ reads } 0) - P(\text{qubit } q \text{ reads } 1),$$

where $x_q$ is bit $q$ of the bitstring $x$. So the expectation is over the random measurement outcome. A value of $+1$ means qubit $q$ is certainly 0, $-1$ means it is certainly 1, and $0$ means it's a fair coin. This is also how a quantum computer estimates $\langle Z_q\rangle$: it repeats the circuit many times (each repeat is called a **shot**) and averages.

**As linear algebra.** $Z_q$ is a diagonal $2^n\times 2^n$ matrix. Its diagonal entry is $+1$ for bitstrings whose bit $q$ is 0 and $-1$ for bitstrings whose bit $q$ is 1. Then

$$\langle Z_q\rangle = \langle\psi|Z_q|\psi\rangle = \psi^\dagger Z_q\,\psi ,$$

and multiplying this out gives the same sum as above. It's the same angle bracket as in the Notation box, applied to the state at the end of the circuit.

For example, take two qubits, write basis states as $|x_1x_0\rangle$, and let $|\psi\rangle = a|00\rangle + b|01\rangle + c|10\rangle + d|11\rangle$. Then

$$Z_0 = \mathrm{diag}(1,-1,1,-1),\qquad Z_1 = \mathrm{diag}(1,1,-1,-1),$$

$$\langle Z_0\rangle = |a|^2-|b|^2+|c|^2-|d|^2,\qquad \langle Z_1\rangle = |a|^2+|b|^2-|c|^2-|d|^2 .$$

The function `expZ(psi, q)` in `otoc_tools.py` computes exactly this: the probability that qubit $q$ reads 0 minus the probability that it reads 1.

> **Key idea.** A circuit produces a whole statevector, but an OTOC only needs $\langle Z_q\rangle$, one number per qubit. That matters: at 50 qubits the statevector has $2^{50}\approx 10^{15}$ amplitudes, far too many to store, while a quantum computer can estimate $\langle Z_q\rangle$ by repeating the circuit and averaging.

Now run the echo with and without the butterfly. Each line prints $\langle Z_q\rangle$ for $q = 0, 1, \dots, 5$. Without the butterfly, $U^\dagger U = I$ brings the state back to $|000000\rangle$, so every qubit certainly reads 0 and every value is 1. With the butterfly, the qubits it reached no longer come back cleanly, and their values drop below 1. Those values are OTOC(1).

In [ ]:
# Check the echo: without the butterfly everything returns to |0>, with it, it doesn't.
psi = run_Udag(run_U(zero(n), gates), gates)                 # U then U^dag, no kick
no_kick = [expZ(psi, q) for q in range(n)]
print("no butterfly   <Z_q> =", np.round(no_kick, 3))
print("with butterfly <Z_q> =", np.round(exact_otoc1(n, gates, b), 3), "  <- this is OTOC(1) at each q")

### ✏️ Exercise: run the echo yourself

Using only `zero`, `run_U`, `apply1`, `run_Udag`, `expZ` and `X` from `otoc_tools`, compute OTOC(1)
at every qubit for the circuit `gates` above (n=6, b=0) and store the list in `my_otoc1`.
The check compares with `exact_otoc1`.

In [ ]:
my_otoc1 = None   # YOUR CODE: psi = ... ; [expZ(psi, q) for q in range(n)]

# --- check (don't edit) ---
checks.close("my_otoc1", my_otoc1, exact_otoc1(n, gates, b), tol=1e-10)

Without the butterfly every qubit returns to $+1$, as expected. With it, qubit 0 flips and its neighbours are disturbed, but qubits 4 and 5 still read exactly $+1$. Four layers weren't enough for the butterfly to reach them.

---
# Part 2: Watching the butterfly spread

One brickwork layer can carry information at most one qubit further along the row. So after $d$ layers the butterfly can only have reached qubits within about $d$ steps of where it started. That boundary is called the **light cone**.

The cell below measures OTOC(1) at every qubit, for every depth from 0 to 20, on a row of 10 qubits. Each value is an average over 8 random circuits.

In [ ]:
n, b, reps = 10, 0, 8
depths = np.arange(0, 21)
heat = np.array([[np.mean([exact_otoc1(n, brickwork(n, d, rng), b)[q] for _ in range(reps)])
                  for q in range(n)] for d in depths])

plt.figure(figsize=(7, 3.6))
plt.imshow(heat.T, origin="lower", aspect="auto", cmap="RdBu", vmin=-1, vmax=1,
           extent=[depths[0] - .5, depths[-1] + .5, -.5, n - .5])
plt.colorbar(label="OTOC(1)")
plt.plot(depths, np.clip(depths, 0, n - 1), "k--", lw=1, label="light cone")
plt.xticks(depths[::2]); plt.yticks(range(n))
plt.xlabel("circuit depth d"); plt.ylabel("probe qubit q"); plt.legend(loc="lower right", fontsize=8)
plt.title("The butterfly (X on qubit 0) spreading along the chain"); plt.grid(False); plt.show()

Each column is one depth. Reading it from top to bottom:

* Above the dashed line (dark blue, value 1) the butterfly hasn't arrived, so there's nothing to learn there.
* Along the dashed line the value starts to drop. Here the answer depends on the details of the particular circuit.
* Well below the line (white, value near 0) everything is scrambled. Every random circuit gives about the same value, within roughly $2^{-n/2}$.

The dashed line is the edge of the light cone. As the depth grows it moves outward by about one qubit per layer, like a wave front, so we'll also call it the butterfly's **front**.

> **Key idea.** The interesting region is the band just behind the front: qubits the butterfly reached only a few layers ago, which sit just below the dashed line.
> - Far ahead of the front the answer is exactly 1, and far behind it about 0, so neither tells you anything about the particular circuit.
> - If you look at every qubit of one circuit, you'll see all three: values close to 1, values close to 0, and the circuit-specific ones in between.

Other layouts behave the same way in spirit. On a 2D grid, for example, the butterfly can spread in several directions at once, so the front moves out faster and reaches more qubits per layer than on a row.

---
# Part 3: OTOC(2), the hard one

There is also a second-order OTOC, **OTOC(2)**:

$$\text{OTOC(2)} = \big\langle (B(t)\,M)^4 \big\rangle .$$

It has eight operators instead of four. The same trick as in Part 1 turns it into a circuit that uses $U$ twice and $U^\dagger$ twice:

### ✏️ Exercise: predict the light cone exactly

A qubit $q$ can only show OTOC(1) $\neq 1$ if the support of $B(t) = U^\dagger X_b U$ reaches it.
In the Heisenberg picture $B$ is conjugated by the **last** layer first, so walk the layers
**backwards**: start with `support = {b}`; for each layer from last to first, every gate on a pair
$(i, i+1)$ that touches the support adds both qubits.

Write `light_cone(n, b, depth)` returning that set. Layer $t$ uses pairs starting at $i = t \bmod 2$
(see `brickwork` in `otoc_tools.py`). The check also confirms every qubit **outside** your set has
OTOC(1) exactly 1 on random circuits.

<details><summary>Hint (click)</summary>

For b=5, depth 1: layer 0 has pairs (0,1),(2,3),(4,5),... so the support becomes {4, 5}.

</details>

In [ ]:
def light_cone(n, b, depth):
    raise NotImplementedError   # YOUR CODE

# --- check (don't edit) ---
def _ref(n, b, depth):
    s = {b}
    for t in reversed(range(depth)):
        for i in range(t % 2, n - 1, 2):
            if i in s or i + 1 in s: s |= {i, i + 1}
    return s
cases = [((10, 0, d), _ref(10, 0, d)) for d in (0, 1, 2, 3, 5)] + [((10, 5, d), _ref(10, 5, d)) for d in (1, 2, 4)]
ok = checks.fn("light_cone", light_cone, cases)
if ok:
    r = np.random.default_rng(5)
    for (nn, bb, dd) in [(10, 0, 4), (10, 5, 3), (9, 3, 2)]:
        o1 = exact_otoc1(nn, brickwork(nn, dd, r), bb)
        outside = [q for q in range(nn) if q not in light_cone(nn, bb, dd)]
        assert np.allclose(o1[outside], 1), "outside the cone OTOC(1) should be 1"
    print("   and outside the cone OTOC(1) == 1 on random circuits, as predicted")

In [ ]:
# The OTOC(2) echo: the OTOC(1) echo twice, with the probe Z_q in between.
draw(echo_schematic(n=6, b=0, q=5, order=2))

So why is OTOC(2) so much harder to simulate than OTOC(1)? Write $B(t)$ as a sum of Pauli strings, $B(t) = \sum_P a_P\,P$.

For OTOC(1), the squares $a_P^2$ are all positive and add up to 1, so they behave like **probabilities**. OTOC(1) then measures, roughly, the probability that the butterfly has reached qubit $q$. Probabilities can be estimated by random sampling, which is why good approximations for OTOC(1) exist.

For OTOC(2), the answer is a sum of products of four coefficients, and those products come with **plus and minus signs**. Large terms can cancel each other and small terms can add up, the same way waves interfere. There's no probability to sample from (physicists call this a **sign problem**), so throwing away the "small" terms can ruin the answer.

> **Key idea.** OTOC(1) asks how much of the butterfly reached $q$. OTOC(2) asks how the different paths to $q$ interfere. That interference is what makes OTOC(2) hard for classical computers, and a good candidate for showing a quantum computer can do better.

The cell below compares OTOC(1) and OTOC(2) as the depth grows, probing the last qubit of a 10-qubit row.

In [ ]:
n, b, q, reps = 10, 0, 9, 25
depths = np.arange(6, 27, 2)
o1, o2 = [], []
for d in depths:
    vals = []
    for _ in range(reps):
        g = brickwork(n, d, rng)
        vals.append((exact_otoc1(n, g, b)[q], exact_otoc2(n, g, b, q)))
    vals = np.array(vals); o1.append(vals[:, 0]); o2.append(vals[:, 1])
o1, o2 = np.array(o1), np.array(o2)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for k, (v, name) in enumerate([(o1, "OTOC(1)"), (o2, "OTOC(2)")]):
    ax[0].errorbar(depths + .15 * k, v.mean(1), v.std(1), marker="o", capsize=3, label=name)
    ax[1].plot(depths, v.std(1), marker="o", label=name)
ax[0].set(xlabel="depth d", ylabel="value (mean ± std)", title=f"Average value over {reps} circuits (q={q}, b={b})")
ax[1].set(xlabel="depth d", ylabel="std across circuits", title=f"How much the value differs between the {reps} circuits")
ax[1].axhline(2 ** (-n / 2), color="gray", ls=":", label=r"noise floor $2^{-n/2}$ (random-noise level)")
for a in ax: a.legend(); a.set_xticks(depths)
plt.tight_layout(); plt.show()

For each depth, the cell draws 25 random circuits and computes OTOC(1) and OTOC(2) at qubit 9 for every one of them. That gives 25 numbers per depth for each OTOC. The two plots summarise those 25 numbers in two different ways.

**Left plot: the average.** Each dot is the mean of the 25 values, and the error bar shows their spread. It has three regimes:

* **Shallow** (depth 8 or less): the butterfly hasn't reached qubit 9 yet, so both OTOCs are exactly 1 for every circuit.
* **Deep**: every circuit acts like a completely random unitary, and both OTOCs shrink to about 0.
* **In between**: the values fall from 1 towards 0. This range of depths is the **window**.

**Right plot: how different the circuits are from each other.** Each dot is the standard deviation (std) of the same 25 values. It answers one question: if I pick a different random circuit at the same depth, how much does the answer change?

* **Std = 0** (depths 6 and 8): every circuit gives the same answer, 1. There's nothing to compute: the answer is known in advance.
* **Large std** (the peak, around depths 10 to 16): each circuit has its own answer, and the only way to find it is to actually work out that circuit. This is the window seen from a different angle.
* **Std at the dotted line** (deep circuits): the circuits still differ a little, by about $2^{-n/2}$, but only by tiny amounts that look like random noise. An approximate method has no structure to latch onto there, and at 50 qubits the differences are far too small to measure anyway.

So the right plot shows where there's a real, circuit-specific signal to compute. For OTOC(2) the peak is taller but narrower: the circuits differ a lot right after the butterfly arrives, and by about depth 18 the spread has already dropped to the noise floor. OTOC(1) keeps a smaller signal for longer.

> **Key idea.** The part of an OTOC worth computing is the part specific to the circuit, where the right plot is high. That's also where to test an approximate method. Outside the window every circuit gives about the same answer, so even a method that knows nothing about the circuit looks accurate there. Part 6 shows how to measure whether estimates follow the differences between circuits.

### ✏️ Exercise: build the OTOC(2) echo yourself

Write `my_otoc2(n, gates, b, q)` with the statevector helpers: run the OTOC(1) echo, apply $Z_q$,
run the echo again, and return $\langle Z_q\rangle$. Compare with `exact_otoc2`.

In [ ]:
def my_otoc2(n, gates, b, q):
    raise NotImplementedError   # YOUR CODE

# --- check (don't edit) ---
r = np.random.default_rng(9)
cases = []
for (nn, dd, bb, qq) in [(6, 4, 0, 3), (8, 6, 2, 7), (7, 5, 0, 1)]:
    g = brickwork(nn, dd, r)
    cases.append(((nn, g, bb, qq), exact_otoc2(nn, g, bb, qq)))
checks.fn("my_otoc2", my_otoc2, cases, tol=1e-10)

### ✏️ Exercise: the butterfly's Pauli weights are probabilities

Part 3 says: write $B(t) = \sum_P a_P P$; then the $a_P^2$ add up to 1, like probabilities.
For $n=6$, depth 3, $b=0$, build the matrix $B(t) = U^\dagger X_0 U$ (use `qiskit.quantum_info.Operator(to_qiskit(n, g)).data`)
and decompose it with `SparsePauliOp.from_operator`. Set:

- `total_weight` = $\sum_P a_P^2$ (expect 1),
- `reached` = a list, for each qubit $q$, of the total weight $\sum a_P^2$ over strings with $X$ or $Y$ on $q$:
  "how much of the butterfly has reached $q$ (in a way $Z_q$ can detect)". It should be 0 outside the light cone.

In [ ]:
from qiskit.quantum_info import Operator, SparsePauliOp
n6, g6 = 6, brickwork(6, 3, np.random.default_rng(3))
total_weight = None   # YOUR CODE
reached = None

# --- check (don't edit) ---
checks.close("total_weight", total_weight, 1.0, tol=1e-9)
checks.truthy("reached is 0 outside the light cone",
              reached is not None and all(abs(reached[q]) < 1e-9 for q in range(6) if q not in _ref(6, 0, 3)),
              hint="label character for qubit q is L[n-1-q]", todo=reached is None)
if reached: print("weight that reached each qubit:", np.round(reached, 3))

---
# Part 4: Exact and approximate simulation

So far every circuit has been small enough for a **statevector** simulation, which stores all $2^n$ complex amplitudes of the state. That's instant at 12 qubits and takes seconds at 20 with the simple simulator in `otoc_tools.py`. The memory doubles with every qubit you add, so at a few dozen qubits it decides whether a statevector is possible at all. The cell below shows how fast it grows. Compare it with what a big GPU holds (tens of GB).

In [ ]:
for n in [12, 20, 30, 40, 50]:
    print(f"statevector memory at n={n}: {16 * 2**n / 1e9:,.3g} GB")

## Why approximate simulators have trouble

For larger circuits you'll need an approximate simulator. A common one is the **matrix product state**, or **MPS**. Instead of one list of $2^n$ amplitudes, an MPS stores a small block of numbers (a **tensor**) for each qubit, linked in a chain along the row. Each link between neighbouring qubits is called a **bond**, and its size is the **bond dimension** $\chi$. Memory then grows like $n\chi^2$ instead of $2^n$.

To see what $\chi$ controls, imagine cutting the row between two qubits, into a left part and a right part. Any state can be written as a sum of terms, each one a state of the left part times a state of the right part. If the two parts aren't entangled, one term is enough. The more entangled they are, the more terms you need. The bond at the cut can hold at most $\chi$ of these terms. If the state needs more, the MPS keeps the $\chi$ biggest and drops the rest, and that is where its error comes from.

The **entanglement entropy** $S$ measures how entangled the two parts are, in bits. Each bit is worth about one Bell pair shared across the cut, and every extra Bell pair doubles the number of terms needed. So an accurate MPS needs $\chi \approx 2^S$. The worst case is a cut through the middle of the row, where $S$ can reach $n/2$. Then $\chi \approx 2^{n/2}$, and $n\chi^2 \approx n\,2^n$ is more memory than the full statevector, so the MPS saves nothing.

The cell below tracks the entanglement across the middle of a 12-qubit row at each stage of an OTOC(2) echo.

In [ ]:
def half_entropy(psi):
    n = psi.ndim
    s = np.linalg.svd(psi.reshape(2 ** (n // 2), -1), compute_uv=False) ** 2
    s = s[s > 1e-14]
    return float(-(s * np.log2(s)).sum())

n, d, b, q = 12, 12, 0, 11
g = brickwork(n, d, rng)
stages, psi = [("start", zero(n))], zero(n)
psi = run_U(psi, g);                      stages.append(("after U", psi))
psi = run_Udag(apply1(psi, X, b), g);     stages.append(("then B U†", psi))
psi = run_U(apply1(psi, Z, q), g);        stages.append(("then M U", psi))
psi = run_Udag(apply1(psi, X, b), g);     stages.append(("end", psi))
S = [half_entropy(p) for _, p in stages]
plt.figure(figsize=(6, 3)); plt.bar([k for k, _ in stages], S, color="#2F6BF0")
plt.axhline(n // 2, color="gray", ls=":", label="maximum possible")
plt.ylabel("half-chain entropy (bits)"); plt.title("Entanglement along the OTOC(2) echo"); plt.legend(); plt.show()
print("bond dimension needed ~ 2^S:", [int(2 ** s) for s in S])

Right after $U$ the state is close to the maximum possible entanglement, and running $U^\dagger$ only removes part of it. An MPS has to hold that peak, so its bond dimension has to be large at exactly the hardest point. On top of that, any error the MPS makes on the way forward doesn't get undone on the way back.

> **Key idea.** Turning up $\chi$ is the natural first thing to try, but the cost grows like $n\chi^2$. For deep enough circuits, no affordable $\chi$ is enough.

## Pauli paths: approximating the operator instead

A second kind of approximation works in the Heisenberg picture from Part 1. Instead of the state, it tracks the measured operator $Z_q$ as a sum of Pauli strings, $\sum_P a_P P$, and pushes it backward through the circuit one gate at a time. Each non-Clifford gate splits a string into several, so the sum keeps growing. To keep it manageable, any term with $|a_P|$ below a **truncation threshold** is dropped after every gate. A smaller threshold keeps more terms, which makes the answer more accurate but the run slower.

Part 3 tells you what to expect. For OTOC(1) the answer is a sum of squares, so dropping small terms gives a small error that always points the same way. For OTOC(2) the dropped terms were interfering with the ones you kept, so the error can go either way. There's one more catch. Run on the full echo, the operator spreads out through $U^\dagger$ and then has to *refocus* through $U$ to give the answer. The terms that would have refocused can look small partway through, and truncation removes exactly those.

> **Key idea.** Pauli paths and MPS fail in different ways. MPS struggles when the state is highly entangled, and Pauli paths struggle when the operator spreads over many strings. Try both on circuits where you know the answer before trusting either one.

---
# Part 5: Running the simulators (offline edition)

BlueQubit offers `cpu`/`gpu` (exact statevector), `mps.cpu`/`mps.gpu` (option `mps_bond_dimension`)
and `pauli-path` (option `pauli_path_truncation_threshold`, smallest allowed `1e-5`). Here `LocalBQ`
plays all three with the same call. On the real platform `cpu`, `mps.cpu` and `pauli-path` are free;
GPU jobs use credits.

> **Note: qubit order.** BlueQubit and Qiskit write Pauli labels with qubit 0 as the **rightmost**
> character, so $Z_0$ on 3 qubits is `"IIZ"`. Getting this backwards gives you the value for the wrong
> qubit without any error message. Use `z_label(q, n)` and you never have to think about it.

The cells below use a 16-qubit circuit built right here, small enough for an exact answer. First,
check that the "platform" gives the same answers as `otoc_tools`' exact simulator.

In [ ]:
def bq_expz(bq, circuit, q, device="cpu", **options):
    # <Z_q> on the (offline) platform. Same code works with bq = bluequbit.init(token).
    r = bq.run(circuit, device=device, pauli_sum=[(z_label(q, circuit.num_qubits), 1.0)], options=options or None)
    ev = r.expectation_value
    return float(ev[0] if isinstance(ev, (list, tuple)) else ev)

n, d, b, q = 16, 12, 0, 7
g = brickwork(n, d, np.random.default_rng(16))
U = to_qiskit(n, g)
tests = [("OTOC1", echo_circuit(U, b, order=1), float(exact_otoc1(n, g, b)[q])),
         ("OTOC2", echo_circuit(U, b, q, order=2), exact_otoc2(n, g, b, q))]
for kind, qc, exact in tests:
    print(f"{kind} at q={q}: exact {exact:+.6f}   platform cpu {bq_expz(bq, qc, q):+.6f}")

Now MPS at several bond dimensions (BlueQubit's version of this cell cost about $5 of GPU credits; offline it's free).

In [ ]:
import time
chis = [2, 4, 8, 16, 32, 64, 128]
t0 = time.time()
for kind, qc, exact in tests:
    est = [bq_expz(bq, qc, q, device="mps.cpu", mps_bond_dimension=c) for c in chis]
    plt.semilogx(chis, est, "o-", base=2, label=f"{kind} MPS estimate")
    plt.axhline(exact, ls=":", color=plt.gca().lines[-1].get_color(), label=f"{kind} exact")
plt.xlabel("bond dimension chi"); plt.ylabel("<Z_q>"); plt.title("MPS vs exact (16 qubits, depth 12)"); plt.legend(); plt.show()
print(f"{time.time() - t0:.0f}s")

Compare with the figure from BlueQubit's real run (saved with the tutorial):

In [ ]:
from IPython.display import Image, display
display(Image(filename="otoc_bluequbit_plots/mps_vs_exact.png", width=520))

Now Pauli paths at several thresholds, on the random circuit and on a **near-Clifford** circuit
(mostly Clifford gates plus a few $R_z$ rotations), where Pauli paths are expected to do well.

In [ ]:
q_nc = 5
U_nc = near_clifford_circuit(n, d, np.random.default_rng(3), p=0.25)
tests_nc = [("OTOC1", echo_circuit(U_nc, b, order=1)), ("OTOC2", echo_circuit(U_nc, b, q_nc, order=2))]
tests_nc = [(kind, qc, simulate_expz(qc, q_nc)) for kind, qc in tests_nc]       # exact, by statevector

thresholds = [1e-1, 3e-2, 1e-2, 1e-3]    # smaller keeps more Pauli terms: slower, more accurate
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, title, cases, qq in [(axes[0], "Random circuit (from above)", tests, q),
                             (axes[1], "Near-Clifford circuit", tests_nc, q_nc)]:
    for kind, qc, exact in cases:
        est = [bq_expz(bq, qc, qq, device="pauli-path", pauli_path_truncation_threshold=t) for t in thresholds]
        ax.semilogx(thresholds, est, "o-", label=f"{kind} Pauli-path estimate")
        ax.axhline(exact, ls=":", color=ax.lines[-1].get_color(), label=f"{kind} exact")
    ax.invert_xaxis()
    ax.set(xlabel="truncation threshold", title=title); ax.legend(fontsize=8)
axes[0].set_ylabel("<Z_q>"); plt.tight_layout(); plt.show()
display(Image(filename="otoc_bluequbit_plots/pauli_path_vs_exact.png", width=700))

On the random circuit, Pauli paths return 0 at every threshold, for both OTOCs. The blue points are hidden under the orange ones. This is the failure described in Part 4: every gate splits every Pauli string, and truncation throws away the terms that would have recombined on the way back.

On the near-Clifford circuit, the same method works. At a loose threshold the estimate is off, and as the threshold shrinks it settles on the exact value. The kind of circuit decides which method works, not only its size.

### ✏️ Exercise: calibrate a method on a known circuit, then apply it

This is the hackathon workflow in miniature. On the **near-Clifford** circuit `U_nc` (n=16, b=0):

1. compute the exact OTOC(2) at every probe qubit $q$ (`simulate_expz(echo_circuit(U_nc, b, q, order=2), q)`), store as `truth_nc`;
2. estimate the same with Pauli paths at threshold `1e-2`, store as `est_nc`;
3. score with `score(est_nc, truth_nc)` and store the dict in `sc_nc`.

You should find a high correlation $r$. Then try MPS with $\chi = 8$ on the same 16 probes if you like.

In [ ]:
truth_nc = None   # YOUR CODE
est_nc = None
sc_nc = None

# --- check (don't edit) ---
checks.truthy("truth_nc has 16 values", truth_nc is not None and len(truth_nc) == 16, todo=truth_nc is None)
checks.truthy("Pauli paths track the near-Clifford OTOC(2) (r > 0.9)", sc_nc is not None and sc_nc["r"] > 0.9,
              hint="threshold 1e-2 on device='pauli-path'", todo=sc_nc is None)
if sc_nc: print({k: round(v, 3) for k, v in sc_nc.items()})

> **Note: running many jobs.** `bq.run(..., asynchronous=True)` returns right away with a job object, and `bq.wait(job)` collects the result later. Submit a whole sweep at once this way, then collect it, rather than waiting on each job in turn.

---
# Part 6: Judging an approximation

## How well do estimates track the truth?

Suppose you've estimated a set of OTOCs, one per circuit or one per qubit, with an approximate method, and you have the exact values for comparison. A natural measure is the correlation $r$ between your estimates and the exact values. It's often reported as a signal-to-noise ratio:

$$r = \mathrm{corr}(\text{your estimates}, \text{exact values}), \qquad \mathrm{SNR} = \frac{r}{\sqrt{1-r^2}} .$$

Since it's a correlation, what counts is getting **the differences between circuits** right. Here is how four kinds of guesses do on a toy set of 40 random 10-qubit circuits, all at the same depth inside the OTOC(2) window.

In [ ]:
n, d, b, q = 10, 12, 0, 9
truth = np.array([exact_otoc2(n, brickwork(n, d, rng), b, q) for _ in range(40)])
guesses = {
    "random numbers":                 rng.uniform(-1, 1, truth.size),
    "the correct average every time":  np.full(truth.size, truth.mean()),
    "exact plus noise, std 0.15":   truth + rng.normal(0, .15, truth.size),
    "exact plus noise, std 0.03": truth + rng.normal(0, .03, truth.size),
}
for name, g in guesses.items():
    sc = score(g, truth)
    print(f"{name:32s}  r = {sc['r']:+.3f}   SNR = {sc['snr']:6.2f}")

The constant guess gets $r = 0$ even though it's the correct average.

> **Key idea.** A correlation only measures whether you know which circuits come out **above** the average and which come out **below**. Small errors are fine as long as they're smaller than the differences between circuits.

### ✏️ Exercise: what the score rewards

Using `truth` from the cell above (40 exact OTOC(2) values), build three estimate arrays and score them:

- `est_affine` = `0.3 * truth + 0.5` (wrong scale **and** offset),
- `est_flipped` = `-truth`,
- `est_rank` = the **ranks** of `truth` (0 for the smallest, 39 for the largest; `np.argsort(np.argsort(truth))`).

Store `score(...)` dicts as `sc_affine`, `sc_flipped`, `sc_rank`. Predict first: which gets $r = 1$?
Which gets SNR 0? (Negative $r$ counts as 0.)

In [ ]:
sc_affine = None   # YOUR CODE
sc_flipped = None
sc_rank = None

# --- check (don't edit) ---
checks.close("affine r", None if sc_affine is None else sc_affine["r"], 1.0, tol=1e-9,
             hint="correlation ignores scale and offset")
checks.close("flipped snr", None if sc_flipped is None else sc_flipped["snr"], 0.0, tol=1e-9,
             hint="negative r counts as 0")
checks.truthy("rank-only estimate scores well (r > 0.8)", sc_rank is not None and sc_rank["r"] > 0.8, todo=sc_rank is None)

## Takeaways for the hackathon

1. **What you submit:** OTOC(2) $=\langle Z_q\rangle$ at the end of the echo U, X_b, U†, Z_q, U, X_b, U†, one number per probe qubit.
2. **Where the signal is:** just behind the butterfly's front. Far ahead it's exactly 1, far behind ≈ 0.
   Your `light_cone()` tells you which qubits are trivially 1 **for free**.
3. **Score = correlation:** scale and offset don't matter; ordering does. Getting the *trivial* qubits right
   (exact 1s vs ≈0s) already buys correlation. Spend compute on the band in between.
4. **Methods fail differently:** MPS needs $\chi\approx 2^S$ at the most entangled point of the echo;
   Pauli paths collapse to 0 on random circuits but shine on near-Clifford ones. Calibrate both on the
   samples (which come with exact answers) before trusting either on hidden circuits.

**Next:** `05_mock_challenge.ipynb`, a full dress rehearsal with hidden circuits, a budget, and a CSV to score.